---
title: "2.17. Formatowanie wyników: kable i kableExtra"
subtitle: Czytelne tabele w raportach HTML
---

# 2.17. Formatowanie wyników: kable i kableExtra

Tabela w raporcie powinna odpowiadać na konkretne pytanie. Czytelnik musi wiedzieć, co oznaczają wiersze, w jakich jednostkach podano wartości i jak potraktowano braki. W tym podrozdziale zamienimy wyniki obliczeń w tabelę z podpisem, uporządkowanymi nagłówkami i oszczędnym wyróżnieniem najważniejszych informacji.

Materiał rozwija tematykę wskazanej winiety [Hao Zhu, *Create Awesome HTML Table with knitr::kable and kableExtra*](https://cran.r-project.org/web/packages/kableExtra/vignettes/awesome_table_in_html.html). `knitr::kable()` tworzy tabelę, a `kableExtra` dodaje warstwy formatowania, m.in. style, nagłówki grupowe i przypisy. Winieta korzysta z `kbl()`, wygodnej nakładki na `kable()`; tutaj zaczniemy od jawnego `knitr::kable()`.

Podrozdział jest samodzielny: dane definiujemy poniżej. Wymagany jest R co najmniej 4.1 oraz pakiety `knitr` i `kableExtra`. Przykłady dotyczą **HTML**; przenoszenie do PDF wymaga wyboru formatu LaTeX i odpowiednich opcji. Czytanie gotowego e-booka nie wymaga instalacji pakietów.

## Przygotowanie i wyświetlanie tabel

Jeśli pakietów brakuje, wykonaj **jednorazowo w konsoli** `install.packages(c("knitr", "kableExtra"))`. Następnie uruchamiaj komórki po kolei. Funkcja pomocnicza poniżej pokazuje wynik HTML w notebooku R. W R Markdown zwykle wystarczy pozostawić obiekt tabeli jako ostatnie wyrażenie komórki. Konsola może wyświetlać kod HTML zamiast wyglądu tabeli — to nie błąd obliczeń.

In [1]:
library(knitr)
library(kableExtra)

pokaz_tabele <- function(tabela) {
  if (requireNamespace("IRdisplay", quietly = TRUE) &&
      isTRUE(getOption("jupyter.in_kernel"))) {
    IRdisplay::display_html(as.character(tabela))
  } else {
    print(tabela)
  }
  invisible(tabela)
}

**Zadanie K.1.** Sprawdź wersje pakietów przez `packageVersion("knitr")` i `packageVersion("kableExtra")`. Wyjaśnij różnicę między tabelą jako wynikiem w przeglądarce a jej kodem HTML.

## Od danych do wyników

Przygotujemy podsumowanie fikcyjnych wyników dwóch grup studentów. `n` oznacza liczbę wszystkich osób, a `n_wynik` liczbę dostępnych wyników. Rozróżnienie jest ważne, bo jedna osoba nie ma wpisanych punktów. Maksymalna możliwa liczba punktów wynosi 100.

In [2]:
oceny <- data.frame(
  grupa = rep(c("A", "B"), each = 4),
  punkty = c(65, 78, 82, NA, 55, 70, 90, 85)
)

podsumowanie <- do.call(rbind, lapply(split(oceny, oceny$grupa), function(d) {
  data.frame(
    grupa = d$grupa[1],
    n = nrow(d),
    n_wynik = sum(!is.na(d$punkty)),
    srednia = mean(d$punkty, na.rm = TRUE),
    odchylenie = sd(d$punkty, na.rm = TRUE)
  )
}))
rownames(podsumowanie) <- NULL
podsumowanie

  grupa n n_wynik srednia odchylenie
1     A 4       3      75   8.888194
2     B 4       4      75  15.811388


Obliczenia wykonujemy na liczbach. Zaokrąglenia, etykiety i kolory dodamy dopiero podczas prezentacji. Dzięki temu późniejsze analizy nadal wykorzystują pełną precyzję.

**Zadanie K.2.** Sprawdź, czy obie grupy mają po 4 osoby, ale odpowiednio 3 i 4 znane wyniki. Oblicz ręcznie średnią grupy A i porównaj ją z ramką `podsumowanie`.

## Pierwsza tabela z kable

Zacznij od podpisu odpowiadającego na pytanie „co przedstawia tabela?”. Nazwy kolumn powinny być zrozumiałe bez czytania skryptu. Liczby wyrównamy do prawej, a etykiety do lewej.

In [3]:
tabela_podstawowa <- knitr::kable(
  podsumowanie,
  format = "html",
  caption = "Wyniki ćwiczenia według grupy (skala 0–100 pkt)",
  col.names = c("Grupa", "Osoby", "Znane wyniki", "Średnia [pkt]", "SD [pkt]"),
  row.names = FALSE,
  align = c("l", "r", "r", "r", "r"),
  digits = c(0, 0, 0, 1, 1),
  format.args = list(decimal.mark = ",", big.mark = " ")
)
pokaz_tabele(tabela_podstawowa)

Grupa,Osoby,Znane wyniki,Średnia [pkt],SD [pkt]
A,4,3,75,"8,9"
B,4,4,75,"15,8"


`digits` steruje zaokrągleniem kolumn liczbowych; nie zmienia ramki źródłowej. Nie gwarantuje dopisania zer do jednakowej liczby miejsc po przecinku. Gdy potrzebujesz np. zawsze dwóch miejsc, użyj `formatC(x, format = "f", digits = 2)` w **kopii do prezentacji** — wynik będzie tekstem. Parametr `row.names = FALSE` usuwa techniczne nazwy wierszy, a nie kolumnę identyfikującą grupę.

**Zadanie K.3.** Utwórz wariant tabeli z dwoma miejscami po przecinku i innym podpisem. Sprawdź, czy `podsumowanie$srednia` nadal jest kolumną liczbową. Dlaczego nie należy liczyć średniej z tekstów przygotowanych przez `formatC()`?

## Styl całej tabeli i wyróżnienia

W `kable_styling()` ustawimy szerokość i pozycję. Funkcje `row_spec()` i `column_spec()` pozwalają wyróżnić wybrane części tabeli. W `row_spec()` numer 0 oznacza nagłówek, a 1 pierwszy wiersz danych. Poniższy przykład używa kolorów i obramowania przypisanych bezpośrednio do elementów HTML.

In [4]:
tabela_styl <- tabela_podstawowa |>
  kableExtra::kable_styling(full_width = FALSE, position = "left", font_size = 15) |>
  kableExtra::row_spec(0, bold = TRUE, color = "white", background = "#234E70") |>
  kableExtra::row_spec(2, background = "#EDF4FA") |>
  kableExtra::column_spec(1, bold = TRUE, border_right = TRUE)
pokaz_tabele(tabela_styl)

Grupa,Osoby,Znane wyniki,Średnia [pkt],SD [pkt]
A,4,3,75,"8,9"
B,4,4,75,"15,8"


W raportach korzystających z Bootstrap można dodać `bootstrap_options = c("striped", "hover", "condensed")` do `kable_styling()`. Wygląd tych klas zależy od załadowanych arkuszy CSS; sam HTML tabeli nie dołącza całego motywu Bootstrap do notebooka ani e-booka. Dlatego przykłady tutaj nie polegają na tych klasach. Kolor nie powinien być jedynym nośnikiem informacji.

**Zadanie K.4.** Zmień kolor nagłówka i wyróżnij pogrubieniem kolumnę średniej. Usuń tło drugiego wiersza. Która wersja jest łatwiejsza do odczytania i dlaczego?

## Nagłówki grupowe i przypisy

Rozdzielimy informacje o liczebności od miar punktów. Liczby podane w `add_header_above()` określają rozpiętość każdej komórki dodatkowego nagłówka: tutaj 1 + 2 + 2 = 5 kolumn. Przypis wyjaśni skrót SD oraz sposób liczenia statystyk.

In [5]:
tabela_opisana <- tabela_styl |>
  kableExtra::add_header_above(c(" " = 1, "Liczebność" = 2, "Punkty" = 2)) |>
  kableExtra::footnote(
    general = c(
      "Dane fikcyjne. Średnią i SD obliczono wyłącznie dla znanych wyników.",
      "SD oznacza odchylenie standardowe próby."
    ),
    general_title = "Uwaga: "
  )
pokaz_tabele(tabela_opisana)

Tabela HTML — wynik kodu powyżej.

Podpis identyfikuje tabelę, nagłówek identyfikuje zmienną, a przypis wyjaśnia szczegóły potrzebne do interpretacji. Nie zastępuj brakujących wyników zerami tylko po to, by tabela wyglądała kompletnie.

**Zadanie K.5.** Dodaj do ramki kolumnę `braki = n - n_wynik`. Zbuduj tabelę ponownie i dostosuj rozpiętość nagłówka „Liczebność” do trzech kolumn. Uzupełnij przypis o informację, w której grupie występuje brak.

## Formatowanie warunkowe

Wyróżnienie powinno wynikać z jawnej reguły. Załóżmy, że chcemy zaznaczyć średnią co najmniej 75 punktów. To umowny próg dydaktyczny, a nie test statystyczny różnicy między grupami. Warunek obliczamy na niezaokrąglonych liczbach, w tej samej kolejności co wiersze tabeli.

In [6]:
osiaga_prog <- podsumowanie$srednia >= 75

tabela_warunkowa <- tabela_opisana |>
  kableExtra::column_spec(
    4,
    bold = osiaga_prog,
    background = ifelse(osiaga_prog, "#DDF0E2", "white"),
    color = "#17212B"
  ) |>
  kableExtra::footnote(
    general = "Zielone tło i pogrubienie: średnia co najmniej 75 pkt (umowny próg).",
    general_title = "Wyróżnienie: "
  )
pokaz_tabele(tabela_warunkowa)

Tabela HTML — wynik kodu powyżej.

Jeśli sortujesz ramkę, wylicz warunek z posortowanych danych. W przeciwnym razie możesz pokolorować inny wynik niż zamierzony. Dla wartości brakujących ustal osobną regułę, np. `!is.na(x) & x >= 75`.

**Zadanie K.6.** Zmień próg na 78 punktów, a następnie na 80. Przewidź, które komórki zostaną wyróżnione. Dopasuj tekst przypisu do każdego progu.

## Grupowanie wierszy i pokazywanie braków

Podsumowanie ma po jednym wierszu na grupę; czasem chcemy pokazać również wyniki poszczególnych osób. `pack_rows()` dodaje etykiety obejmujące kolejne wiersze. Dane muszą wcześniej znajdować się w odpowiedniej kolejności — etykieta grupy sama ich nie sortuje.

In [7]:
szczegoly <- data.frame(
  osoba = paste0("S", seq_len(nrow(oceny))),
  grupa = oceny$grupa,
  punkty = oceny$punkty
)
szczegoly <- szczegoly[order(szczegoly$grupa, szczegoly$osoba), ]
do_pokazania <- szczegoly[c("osoba", "punkty")]
do_pokazania$punkty <- ifelse(is.na(do_pokazania$punkty), "brak", as.character(do_pokazania$punkty))

tabela_grupy <- knitr::kable(
  do_pokazania, format = "html", row.names = FALSE,
  col.names = c("Osoba", "Punkty"), align = c("l", "r"),
  caption = "Indywidualne wyniki ćwiczenia — dane fikcyjne"
) |>
  kableExtra::kable_styling(full_width = FALSE) |>
  kableExtra::pack_rows("Grupa A", 1, 4) |>
  kableExtra::pack_rows("Grupa B", 5, 8)
pokaz_tabele(tabela_grupy)

Tabela HTML — wynik kodu powyżej.

Konwersja punktów na tekst nastąpiła tylko w `do_pokazania`; oryginalna kolumna `oceny$punkty` pozostaje liczbowa. `collapse_rows()` jest innym sposobem ograniczenia powtarzanych etykiet — scala powtarzające się komórki. Przy pierwszych raportach jawne etykiety grup są zwykle łatwiejsze do sprawdzenia.

**Zadanie K.7.** Dodaj osobę z wynikiem 88 do grupy B. Dostosuj zakres grupowania. Wyjaśnij, dlaczego po dodaniu osoby do grupy A trzeba byłoby zmienić również początek zakresu grupy B.

## Duże tabele i HTML w komórkach

Najpierw zdecyduj, czy czytelnik potrzebuje wszystkich wierszy. W tekście raportu często wystarczy podsumowanie, a szczegóły można pokazać w przewijanym obszarze. `scroll_box()` ogranicza jego rozmiar.

In [8]:
tabela_przewijana <- tabela_grupy |>
  kableExtra::scroll_box(width = "100%", height = "220px")
pokaz_tabele(tabela_przewijana)

Tabela HTML — wynik kodu powyżej.

**Zadanie K.8.** Zmień wysokość obszaru na `150px` i porównaj wygodę odczytu. Napisz jedno zdanie wyjaśniające, kiedy lepsze jest podsumowanie zamiast przewijania szczegółów.

Domyślne `escape = TRUE` w `kable()` traktuje specjalne znaki jako tekst. Gdy chcesz celowo umieścić własny HTML, np. wynik `cell_spec()`, potrzebne może być `escape = FALSE`. Stosuj to tylko do kontrolowanej treści; obcego tekstu nie należy bezpośrednio interpretować jako HTML. Do zwykłego kolorowania kolumn wystarczy `column_spec()`.

**Zadanie K.9.** Utwórz jednokolumnową ramkę z napisem `"<b>Wynik</b>"`. Porównaj tabelę z domyślnym `escape = TRUE` oraz z `escape = FALSE`. W którym przypadku znaczniki zostają pokazane dosłownie?

## Zapis tabeli i użycie w raporcie

Obiekt `tabela_warunkowa` zawiera formatowanie, a `podsumowanie` nadal przechowuje dane do obliczeń. Samodzielny plik HTML możesz zapisać poniższym poleceniem w konsoli, w katalogu własnego projektu:

```text
kableExtra::save_kable(tabela_warunkowa, file = "wyniki_grup.html", self_contained = TRUE)
```

Samowystarczalny eksport może wymagać Pandoc. Gdy go brakuje, zainstaluj go lub zastosuj `self_contained = FALSE` i zachowaj przy pliku utworzony katalog zasobów. W notebooku używamy `pokaz_tabele()`. W R Markdown umieść tabelę jako ostatnie wyrażenie komórki; przy ręcznym wypisywaniu HTML przez `cat()` (np. w pętli) ustaw opcję komórki `results = 'asis'`.

**Zadanie K.10 — mały raport.** Utwórz tabelę wyników trzech grup po co najmniej trzy osoby. Dla każdej grupy pokaż liczbę znanych wyników, średnią i SD. Dodaj podpis, jednostki, nagłówek grupowy oraz przypis o brakach. Wyróżnij najwyższą średnią, uwzględniając ewentualny remis. Zapisz tabelę jako HTML i otwórz plik w przeglądarce. Pod tabelą napisz dwa zdania interpretacji; nie wyciągaj wniosków o istotności statystycznej z samego koloru.

## Przed oddaniem raportu

**Zadanie K.11.** Sprawdź własną tabelę: czy ma opisowy podpis, jednostki i wyjaśnienie skrótów? Czy liczebności uwzględniają braki? Czy wyróżnienia odpowiadają opisanej regule? Czy po usunięciu kolorów nadal można zrozumieć wyniki? Czy formatowanie pozostawiło dane źródłowe bez zmian?

Pełny przegląd dodatkowych możliwości, w tym motywów, scalania komórek i miniwykresów, znajdziesz w [winiecie kableExtra](https://cran.r-project.org/web/packages/kableExtra/vignettes/awesome_table_in_html.html). Przykłady w tym podrozdziale wykorzystują własne dane dydaktyczne i bazowy potok `|>`.

[← Wróć do struktur danych, operatorów i funkcji](rozdzial-2.ipynb)

[Przejdź do podrozdziału 2.18: tabele opisowe z gtsummary →](rozdzial-2-gtsummary.ipynb)